# step00d - Stage-change audit: what each module does to the data (DatCnv onwards)

For the CTD processing meeting (Leah's request): quantify, at every stage from DatCnv onward, how
much the data actually change. "Change" is not one thing across the chain, so this audit keeps **two
tracks**:

- **Row track** - which scans are *removed* or *flagged*. DatCnv sets the scan population; Section
  removes the surface soak and equilibration dip; Loop Edit flags heave scans (marks, does not
  delete); Bin Average collapses scans into depth bins. Metric: counts and percentages.
- **Value track** - how much the numbers *move* on the scans that survive. Filter (low-pass),
  Align CTD (time shift), and Cell Thermal Mass (conductivity correction) change values without
  removing rows. Metric: per-variable RMS of (after - before) in physical units, the same
  normalised by the variable's own profile standard deviation (so P, T, C, S compare on one axis),
  and the percent of scans moved beyond the sensor's nominal resolution.

DatCnv is the **baseline**: it converts raw frequencies/voltages to physical units, so there is no
physical-units predecessor to difference against. We describe it (scan count, rate, range, bad
flags), we do not assign it a change magnitude.

Stage-specific "did it do its job" diagnostics, with their reasoning:

| Stage | Row effect | Value metric | Targeted diagnostic |
|-------|-----------|--------------|---------------------|
| DatCnv | sets scans | none (baseline) | n scans, sample rate, P range, bad-flag count |
| Filter | none | RMS/normalised delta on P, T, C | reduction in high-frequency variance (std of dX/dt) |
| Align CTD | few edge scans | (raw delta is mostly the lag, reported but not headline) | **salinity-spike reduction** (the real purpose) |
| Cell Thermal Mass | none | RMS delta on C, and resulting delta S | computed whole-profile AND restricted to the thermocline |
| Section | removes soak+dip | n/a | scans removed, % removed |
| Loop Edit | flags heave | n/a | scans flagged, % flagged |
| Derive | adds S, density, sigma-theta | n/a (new variables) | density-inversion count (static-stability QC) |
| Bin Average | scans to bins | n/a | scans-per-bin, within-bin std (resolution traded for precision) |

**Why the Align metric is salinity spiking, not raw delta-T:** the module only time-shifts the
sensors, so a raw per-variable delta is dominated by the shift itself and says nothing about whether
alignment helped. Alignment exists to make conductivity and temperature coincide in time so derived
salinity stops spiking, so the honest metric is the change in salinity spiking.

**Why CTM is reported whole-profile and at the thermocline:** the thermal-lag error scales with the
temperature gradient (Morison et al. 1994), so it concentrates at the thermocline and is near zero in
the mixed layer. A whole-profile RMS averages it toward zero and hides the effect that matters.

**Honest limitations (state these on the slide):** this audit measures how much each step *moves* the
data, not whether the result is more *correct*. Only bottle salinity/O2 calibration validates
correctness, so language stays at "change", not "improvement". Per-cast change varies strongly with
sea state, so the fleet view is median and range, never a single number. The Bin Average change
depends on bin size (1 m science product here).

Output: a per-cast stage ledger CSV, two detail casts walked through in full (waterfall + value
change), and a fleet summary (median + range across all casts). Diagnostic only: writes figures and
CSVs to `L2/CTD/qc_plots/`, changes no L1/L2 product.

Sources: Sea-Bird SBE Data Processing User Manual (module behaviour, order); Morison et al. 1994,
J. Atmos. Oceanic Technol. 11, 1151-1164 (cell thermal-lag, error scales with dT/dt); UNESCO 1983
(PSS-78 salinity, EOS-80 density); GO-SHIP/WOCE CTD conventions (bin averaging, scans-per-bin,
within-bin std); McRaven and Cedarholm batch SBE processing training (stage order); and the measured
Filter/Section/Loop Edit numbers from step00a and step00b.

---

**What is in this notebook (sections):** 1-2 setup and readers; 3 re-runs the chain capturing every stage; 4 computes the per-stage metrics; 5 the figure builders; 6 two casts in full detail (scan-count waterfall + value-change, saved to `qc_plots/`); 7 the fleet summary across all casts; **8 the interactive explorer: pick any cast and variable and see its profile at every stage with downcast and upcast separated.** 9 is a reproducibility check that compares what the audit reproduces against step01's actual L2 products, stage by stage (prints MATCH when they are identical). Run the cells top to bottom (Restart Kernel + Run All). Nothing here changes your L1/L2 products; it only reads them and writes figures/CSVs.


## 1. Imports, paths, settings, style

In [ ]:
from __future__ import annotations
import re, sys, subprocess
from collections import OrderedDict
from pathlib import Path
import numpy as np
import pandas as pd

# Display backend. INTERACTIVE_ZOOM=True uses the ipympl "widget" backend so the stage explorer is a
# live, mouse drag/scroll zoomable canvas. The STATIC audit figures (waterfall, change, summary) are
# built as detached Agg figures (new_static_fig) and shown as PNGs, so they stay reliable either way.
INTERACTIVE_ZOOM = True    # True = live ipympl zoom canvas with the box-zoom toolbar (default).
                           # If it ever renders blank in your Jupyter, set this to False for a
                           # large static image instead.
import matplotlib
try:
    get_ipython().run_line_magic("matplotlib", "widget" if INTERACTIVE_ZOOM else "inline")
    if INTERACTIVE_ZOOM:
        print("ipympl widget backend active - the stage explorer has mouse drag/scroll zoom.")
except Exception:
    if INTERACTIVE_ZOOM:
        print("ipympl not available -> explorer will be static (no mouse zoom). To enable it:")
        print("    pip install ipympl ipywidgets   then: Restart Kernel + Run All")
    INTERACTIVE_ZOOM = False
    try:
        get_ipython().run_line_magic("matplotlib", "inline")
    except Exception:
        try: matplotlib.use("Agg")
        except Exception: pass
import matplotlib.pyplot as plt
from matplotlib import rcParams
if INTERACTIVE_ZOOM:
    try: plt.ioff()   # stop ipympl auto-displaying a 2nd stray copy of each explorer figure
    except Exception: pass
try:
    from IPython.display import display, Image
except Exception:
    def display(obj): print(obj)
    Image = None

# --------------------------------------------------------------- paths / config
CTD_ROOT   = Path(r"C:\Projects\ctd_pipeline")
CRUISE_ID  = "P45_06"   # <-- the ONLY switch to change per cruise (e.g. "P45_07"); then Run All
CRUISE_DIR = CTD_ROOT / "cruises" / CRUISE_ID

# DatCnv output (L1) is the audit baseline; the chain is re-run from here so every stage is captured.
DATCNV_DIR = CRUISE_DIR / "L1" / "CTD"
SBE_BIN_DIR = Path(r"C:\Program Files (x86)\Sea-Bird\SBEDataProcessing-Win32")
PSA_DIR     = CTD_ROOT / "psa"
WORK        = CTD_ROOT / "_sbe_work" / "audit"; WORK.mkdir(parents=True, exist_ok=True)
QC_PLOT_DIR = CRUISE_DIR / "L2" / "CTD" / "qc_plots"; QC_PLOT_DIR.mkdir(parents=True, exist_ok=True)

# PSA per module (names confirmed from step01_run_sbe_processing_v2.py)
PSA = {
    "filter":   "05_filter.psa",
    "align":    "02_alignctd.psa",
    "celltm":   "04_celltm.psa",
    "loopedit": "06a_loopedit_percentmean.psa",
    "loopedit_shallow": "06b_loopedit_fixedmin.psa",   # shallow casts (<35 m): fixed min velocity
    "derive":   "07_derive.psa",
    "binavg":   "08b_binavg_1m_down.psa",
}

# Two casts walked through in full (one calmer, one rougher if known) + the fleet
DETAIL_CASTS = ["P45_06_CTD_03", "P45_06_CTD_06"]   # preferred; auto-tops-up below for any cruise
ALL_CASTS = sorted(p.name[:-len(".cnv")] for p in DATCNV_DIR.glob("*.cnv")) or list(DETAIL_CASTS)
# Keep only detail casts that exist for THIS cruise; top up to two spread across the fleet otherwise,
# so changing CRUISE_ID above is all it takes to run on P45_07 or any other cruise.
DETAIL_CASTS = [c for c in DETAIL_CASTS if (DATCNV_DIR / f"{c}.cnv").exists()]
if len(DETAIL_CASTS) < 2 and ALL_CASTS:
    for _c in [ALL_CASTS[0], ALL_CASTS[len(ALL_CASTS) // 2], ALL_CASTS[-1]]:
        if _c not in DETAIL_CASTS:
            DETAIL_CASTS.append(_c)
        if len(DETAIL_CASTS) >= 2:
            break

LAT = 5.0           # nominal shelf latitude for pressure->depth (Gulf of Guinea)
BIN_DB = 1.0        # 1 m depth bins (science product) for within-bin stats

# SBE 19plus V2 nominal sensor resolution (datasheet) - used only to count scans "moved beyond
# resolution". Adjust if your config/full-scale differs; pressure res is ~0.002% of full scale.
RES = {"P": 0.01, "T": 0.0001, "C": 0.00005}   # dbar, degC, S/m
C_STD = 4.2914

# Section onset (matches step00b; this value sections correctly - do not change)
# (Section onset uses step01's exact fs-scan window + degeneracy guard, defined with
#  section_to_cnv below, so the audit reproduces step01's sectioning byte-for-byte.)

# ctd_lib (validated PSS-78 salinity + potential temperature); fall back to inline if not importable
sys.path.insert(0, str(CTD_ROOT))
try:
    from ctd_lib import practical_salinity, potential_temperature
    _HAVE_LIB = True
except Exception as e:
    _HAVE_LIB = False
    print("ctd_lib not importable (", e, ") - using inline PSS-78/potential-temp fallback.")

rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200,
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.linewidth": 0.8, "xtick.direction": "in", "ytick.direction": "in",
    "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 7.5, "legend.frameon": True, "legend.framealpha": 0.9,
    "axes.grid": False, "font.family": "DejaVu Sans",
})
# stage colours
STAGE_ORDER = ["datcnv", "filter", "align", "celltm", "section", "loopedit", "derive", "binavg"]
STAGE_LABEL = {"datcnv": "DatCnv", "filter": "Filter", "align": "Align CTD", "celltm": "Cell TM",
               "section": "Section", "loopedit": "Loop Edit", "derive": "Derive", "binavg": "Bin Avg"}
VAR_COLOR = {"P": "#2E6FB0", "T": "#E8703A", "C": "#4C9F70", "S": "#9467BD"}
C_REMOVE, C_FLAG, C_KEEP = "#D62728", "#E8B23A", "#2E6FB0"
C_DOWN, C_UP = "#222222", "#8a8a8a"   # downcast (dark) vs upcast (grey) in the stage explorer


def new_static_fig(nrows=1, ncols=1, *, sharex=False, sharey=False, squeeze=True, gridspec_kw=None, **fig_kw):
    """Detached Agg figure+axes (never a live widget); reliable static PNG output."""
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    fig = Figure(**fig_kw); FigureCanvasAgg(fig)
    return fig, fig.subplots(nrows, ncols, sharex=sharex, sharey=sharey, squeeze=squeeze, gridspec_kw=gridspec_kw)


def finish(fig, out):
    """Save PNG then display it inline. Never raises."""
    try:
        fig.savefig(out); ok = True
    except Exception as e:
        print("    savefig failed:", e); ok = False
    if ok and Image is not None:
        try: display(Image(filename=str(out)))
        except Exception: display(fig)
    return out if ok else None


def _despine(ax):
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


_EXPLORER_FIGS = {}
def show_live(fig, key):
    """Show a live, zoomable ipympl canvas when INTERACTIVE_ZOOM is on (mouse drag box-zoom, scroll
    zoom, toolbar Home to reset), else a static inline PNG. Scales to the output width and closes the
    previous figure for this key so repeated redraws do not pile up."""
    if INTERACTIVE_ZOOM:
        prev = _EXPLORER_FIGS.get(key)
        if prev is not None:
            try: plt.close(prev)
            except Exception: pass
        _EXPLORER_FIGS[key] = fig
        for attr, val in (("header_visible", False), ("footer_visible", False)):
            try: setattr(fig.canvas, attr, val)
            except Exception: pass
        display(fig.canvas)
        try: fig.canvas.draw_idle()        # force the first paint (ioff() suppresses the auto-draw)
        except Exception: pass
    else:
        plt.show(); plt.close(fig)

print("Cruise:", CRUISE_ID, "| detail casts:", DETAIL_CASTS, "| all casts found:", len(ALL_CASTS))

## 2. Readers, salinity, density (EOS-80), metric helpers

In [ ]:
def parse_cnv_raw(path):
    """Parse a .cnv keeping ALL scans (incl. flagged). Returns a numeric DataFrame or None."""
    if path is None: return None
    text = Path(path).read_text(encoding="latin-1", errors="replace").splitlines()
    names, ds = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, ln in enumerate(text):
        s = ln.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(ln)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): ds = i + 1; break
    if ds is None: return None
    cols = [names[k] for k in sorted(names)]
    rows = [l.split()[:len(cols)] for l in text[ds:] if l.strip() and len(l.split()) >= len(cols)]
    df = pd.DataFrame(rows, columns=cols).apply(pd.to_numeric, errors="coerce").reset_index(drop=True)
    # Sea-Bird bad-flag sentinel -> NaN so it never plots or enters a statistic. The sentinel is
    # -9.990e-29 (a TINY negative number, not a large one), which Align CTD etc. write into the record
    # ends they shift out of. Bracket it tightly; real T/C/P/S/O2 values never fall in this range.
    bad = (df.le(-8e-29) & df.ge(-1.2e-28)) | (df.abs() > 1e30)
    # IMPORTANT: the loop-edit "flag" column legitimately uses -9.990e-29 to mark a FLAGGED scan
    # (0 = good). Do not NaN it, or the red flagged points and the flag % would vanish.
    for fc in ("flag",):
        if fc in bad.columns:
            bad[fc] = False
    return df.mask(bad)


def read_cnv_full(path):
    """(header_lines, cols, rows_raw) so a sectioned .cnv can be re-written with its header."""
    txt = Path(path).read_text(encoding="latin-1", errors="replace").splitlines()
    names, ds = {}, None
    nr = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    for i, l in enumerate(txt):
        s = l.strip()
        if s.startswith("#") or s.startswith("*"):
            m = nr.search(l)
            if m: names[int(m.group(1))] = m.group(2).strip()
            if s.upper().startswith("*END*"): ds = i + 1; break
    cols = [names[k] for k in sorted(names)]
    rows_raw = [l for l in txt[ds:] if l.strip()]
    return txt[:ds], cols, rows_raw


def pick(df, *cands):
    if df is None: return None
    for c in cands:
        if c in df.columns: return c
    for c in df.columns:
        if any(x.lower() in c.lower() for x in cands): return c
    return None


def col_P(df): return pick(df, "prdM", "prDM", "prM", "prSM", "pressure")
def col_T(df): return pick(df, "tv290C", "t090C", "tv290", "t190C", "temperature")
def col_time(df): return pick(df, "timeS", "timeJ", "scan")


def cond_Sm(df):
    """Conductivity column as a numpy array in S/m (converts mS/cm -> S/m)."""
    c = pick(df, "c0S/m", "cond0S/m", "c_S/m", "c0mS/cm", "c0ms/cm", "cond0mS/cm", "conductivity")
    if c is None: return None, None
    x = pd.to_numeric(df[c], errors="coerce").to_numpy(float)
    if "ms/cm" in c.lower():
        x = x * 0.1
    return x, c


def flag_mask(df, flag_col="flag"):
    """Boolean mask of loop-edit flagged scans (flag != 0), or None if no flag column."""
    if df is None or flag_col not in df.columns:
        return None
    return pd.to_numeric(df[flag_col], errors="coerce").fillna(0) != 0


def split_down_up(df, pcol):
    """Split a stage DataFrame into (downcast, upcast) at the deepest scan."""
    p = pd.to_numeric(df[pcol], errors="coerce")
    imax = int(p.idxmax()) if p.notna().any() else len(df) - 1
    return df.iloc[:imax + 1], df.iloc[imax:]


def num(df, col):
    return pd.to_numeric(df[col], errors="coerce").to_numpy(float) if (df is not None and col in df.columns) else None


# ---- salinity + density -----------------------------------------------------
if not _HAVE_LIB:
    def _sal78(R, T68, P):
        a=[0.0080,-0.1692,25.3851,14.0941,-7.0261,2.7081]; b=[0.0005,-0.0056,-0.0066,-0.0375,0.0636,-0.0144]
        c=[0.6766097,2.00564e-2,1.104259e-4,-6.9698e-7,1.0031e-9]; d=[3.426e-2,4.464e-4,4.215e-1,-3.107e-3]
        e=[2.070e-5,-6.370e-10,3.989e-15]; R=np.asarray(R,float)
        Rp=1+(P*(e[0]+e[1]*P+e[2]*P*P))/(1+d[0]*T68+d[1]*T68*T68+(d[2]+d[3]*T68)*R)
        rt=c[0]+c[1]*T68+c[2]*T68**2+c[3]*T68**3+c[4]*T68**4; Rt=np.maximum(R/(Rp*rt),1e-12); s=np.sqrt(Rt)
        dS=(T68-15)/(1+0.0162*(T68-15))*(b[0]+b[1]*s+b[2]*Rt+b[3]*Rt*s+b[4]*Rt**2+b[5]*Rt**2*s)
        return a[0]+a[1]*s+a[2]*Rt+a[3]*Rt*s+a[4]*Rt**2+a[5]*Rt**2*s+dS
    def practical_salinity(C_Sm, T90, P):
        return _sal78(np.asarray(C_Sm,float)/C_STD, np.asarray(T90,float)*1.00024, P)
    def _adtg(S,T,P):
        a0,a1,a2,a3=3.5803e-5,8.5258e-6,-6.836e-8,6.6228e-10; b0,b1=1.8932e-6,-4.2393e-8
        c0,c1,c2,c3=1.8741e-8,-6.7795e-10,8.733e-12,-5.4481e-14; d0,d1=-1.1351e-10,2.7759e-12
        e0,e1,e2=-4.6206e-13,1.8676e-14,-2.1687e-16
        return (a0+(a1+(a2+a3*T)*T)*T+(b0+b1*T)*(S-35)+((c0+(c1+(c2+c3*T)*T)*T)+(d0+d1*T)*(S-35))*P+(e0+(e1+e2*T)*T)*P*P)
    def potential_temperature(S,T90,P,PR=0.0):
        S=np.asarray(S,float); T=np.asarray(T90,float)*1.00024; P=np.asarray(P,float); H=PR-P
        XK=H*_adtg(S,T,P); T=T+0.5*XK; Q=XK; P2=P+0.5*H
        XK=H*_adtg(S,T,P2); T=T+0.29289322*(XK-Q); Q=0.58578644*XK+0.121320344*Q
        XK=H*_adtg(S,T,P2); T=T+1.707106781*(XK-Q); Q=3.414213562*XK-4.121320344*Q
        P3=P2+0.5*H; XK=H*_adtg(S,T,P3); T=T+(XK-2.0*Q)/6.0
        return T/1.00024


def _rho_p0(S, T68):
    """EOS-80 density at one atmosphere (p=0), kg/m3. Checked: rho(35,5,0)=1027.675."""
    T = T68
    rw = (999.842594 + 6.793952e-2*T - 9.095290e-3*T**2 + 1.001685e-4*T**3
          - 1.120083e-6*T**4 + 6.536332e-9*T**5)
    A = 8.24493e-1 - 4.0899e-3*T + 7.6438e-5*T**2 - 8.2467e-7*T**3 + 5.3875e-9*T**4
    B = -5.72466e-3 + 1.0227e-4*T - 1.6546e-6*T**2
    Cc = 4.8314e-4
    return rw + A*S + B*S**1.5 + Cc*S**2


def sigma_theta(C_Sm, T90, P):
    """sigma-theta (kg/m3) from conductivity[S/m], T90, P: potential temp to 0 dbar, EOS-80 at p=0."""
    S = practical_salinity(C_Sm, T90, P)
    th90 = potential_temperature(S, T90, P)
    return _rho_p0(S, np.asarray(th90, float) * 1.00024) - 1000.0


def salinity_of(df):
    """Practical salinity from a stage DataFrame (C in S/m, T90, P), or None if columns missing."""
    cS, _ = cond_Sm(df); p = col_P(df); t = col_T(df)
    if cS is None or p is None or t is None: return None
    return practical_salinity(cS, num(df, t), num(df, p))


# ---- metric helpers ---------------------------------------------------------
def value_delta(xb, xa, pressure=None, res=None):
    """Per-variable change: after - before on the common finite scans."""
    n = min(len(xb), len(xa)); xb = np.asarray(xb[:n], float); xa = np.asarray(xa[:n], float)
    d = xa - xb; m = np.isfinite(d) & np.isfinite(xb)
    if m.sum() < 3:
        return dict(n=0, rms=np.nan, nrms=np.nan, mean=np.nan, max_abs=np.nan, p_at_max=np.nan, pct_moved=np.nan)
    dd = d[m]; base_sd = np.nanstd(xb[m])
    out = dict(n=int(m.sum()), rms=float(np.sqrt(np.mean(dd**2))), mean=float(np.mean(dd)),
               max_abs=float(np.max(np.abs(dd))),
               nrms=float(np.sqrt(np.mean(dd**2)) / base_sd) if base_sd > 0 else np.nan,
               pct_moved=float(100*np.mean(np.abs(dd) > res)) if res else np.nan)
    if pressure is not None:
        pr = np.asarray(pressure[:n], float)[m]
        out["p_at_max"] = float(pr[int(np.argmax(np.abs(dd)))]) if pr.size else np.nan
    else:
        out["p_at_max"] = np.nan
    return out


def dxdt_std(x):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    return float(np.nanstd(np.gradient(x))) if x.size > 3 else np.nan


def spike_index(S, w=7):
    """High-frequency salinity wiggle: std of (S - running median). Alignment should lower it."""
    s = pd.Series(np.asarray(S, float))
    resid = s - s.rolling(w, center=True, min_periods=1).median()
    return float(np.nanstd(resid.to_numpy()))


def thermocline_mask(T, P, frac=0.25):
    """Scans in the strongest-gradient band (top `frac` of |dT/dP|)."""
    T = np.asarray(T, float); P = np.asarray(P, float)
    g = np.abs(np.gradient(T, P))
    g = np.where(np.isfinite(g), g, 0.0)
    if not np.any(g > 0): return np.zeros(len(T), bool)
    thr = np.nanquantile(g[g > 0], 1 - frac)
    return g >= thr


def density_inversions(C_Sm, T90, P, thr=0.005):
    """On the downcast (increasing P), count sigma-theta decreases > thr (static instability)."""
    P = np.asarray(P, float); imax = int(np.nanargmax(P)) if np.isfinite(P).any() else 0
    sl = slice(0, imax + 1)
    st = sigma_theta(np.asarray(C_Sm)[sl], np.asarray(T90)[sl], P[sl])
    pr = P[sl]; ok = np.isfinite(st) & np.isfinite(pr)
    st, pr = st[ok], pr[ok]
    if st.size < 5: return 0, np.nan
    order = np.argsort(pr); dd = np.diff(st[order])
    neg = dd[dd < -thr]
    return int(neg.size), (float(-neg.min()) if neg.size else 0.0)

## 3. Re-run the chain, capturing every stage

Starts from the DatCnv L1 cnv (the baseline) and runs Filter, Align CTD, Cell Thermal Mass, Section
(python), Loop Edit, Derive, Bin Average in order, writing each intermediate to the work folder and
parsing it. Command pattern mirrors step01 (`/pPSA /iIN /oOUT /fSTEM /s`).

In [ ]:
def run_sbe(exe, psa_name, in_cnv, out_stem):
    psa = PSA_DIR / psa_name
    if not psa.exists():
        print(f"    MISSING PSA: {psa}  <-- this .psa does not exist (maybe only a .bak). {exe} skipped,"
              f" so this and all later stages are empty.")
        return None
    if in_cnv is None:                                   # upstream stage failed; do not chain onward
        return None
    cmd = [str(SBE_BIN_DIR / exe), f"/p{psa}", f"/i{in_cnv}", f"/o{WORK}", f"/f{out_stem}", "/s"]
    res = subprocess.run(cmd, capture_output=True, text=True)
    if res.returncode != 0:
        print(f"    WARNING {exe} rc={res.returncode}: {(res.stderr or '')[:160]}")
    # find the output regardless of any NameAppend inside the PSA (e.g. a leftover '_1s')
    cands = sorted(WORK.glob(f"{out_stem}*.cnv"), key=lambda q: q.stat().st_mtime, reverse=True)
    return cands[0] if cands else None


def check_psas():
    """Pre-flight: list any required .psa that is missing (the usual cause of empty later stages)."""
    missing = [n for n in PSA.values() if not (PSA_DIR / n).exists()]
    if missing:
        print("MISSING active PSA files (create these - a .bak backup is not loaded):")
        for n in missing:
            print("   ", PSA_DIR / n)
    else:
        print("PSA pre-flight OK: all required .psa present.")
    return not missing


# Section step: IDENTICAL to step01 (section_cnv / _descent_onset) so the audit reproduces the
# production pipeline EXACTLY - same descent onset (fs-scan window + degeneracy guard) and a
# byte-for-byte sectioned .cnv (CRLF preserved, only '# nvalues' updated). Do not change this, or the
# audit would diverge from step01 at Section and every stage after it (Loop Edit, Derive, Bin Avg).
SECTION_SOAK_MIN_DROP_DB = 0.10   # min pressure increase over ~1 s to count as descending (step01)
SECTION_DEFAULT_FS = 24.0         # fallback sample rate if the header has no interval (step01)
# Shallow-cast handling, mirrored from step01 (Leah, Oct 2026). Keep in sync with step01, or the
# reproducibility check will diverge on shallow casts.
SHALLOW_CAST_MAX_DB      = 35.0   # "shallow" if max pressure < this (dbar)
SECTION_SHALLOW_FLOOR_DB = 5.0    # on shallow casts, Section never trims below this depth


def _cnv_pressure_column(header_lines):
    name_re = re.compile(r"#\s*name\s+(\d+)\s*=\s*([^:]+?)\s*[:=]", re.I)
    best = None
    for line in header_lines:
        m = name_re.search(line)
        if not m: continue
        idx = int(m.group(1)); label = m.group(2).strip().lower()
        if label in ("prdm", "prm", "prsm") or label.startswith("prdm"): return idx
        if best is None and "pr" in label and "spar" not in label: best = idx
    return best


def _cnv_sample_rate(header_lines):
    for line in header_lines:
        low = line.lower()
        if "interval" in low and "seconds" in low:
            m = re.search(r"seconds\s*:\s*([0-9.eE+-]+)", line)
            if m:
                try:
                    dt = float(m.group(1))
                    if dt > 0: return 1.0 / dt
                except ValueError: pass
    return SECTION_DEFAULT_FS


def _descent_onset(pressure, fs):
    """step01's descent onset: walk back from the deepest scan over a ~1 s (fs-scan) window; fall
    back to the surface crossing if that is degenerate (too short / <10 db)."""
    p = np.asarray(pressure, float)
    if p.size == 0 or np.isnan(p).all(): return 0
    imax = int(np.nanargmax(p))
    win = max(int(round(fs)), 1)
    if imax < win + 1: return 0
    seg = pd.Series(p[:imax + 1]).rolling(win, center=True, min_periods=1).median().to_numpy()
    fwd = np.full(len(seg), np.nan); fwd[:len(seg) - win] = seg[win:] - seg[:len(seg) - win]
    descending = np.where(np.isnan(fwd), True, fwd > SECTION_SOAK_MIN_DROP_DB)
    i = imax
    while i > 0 and descending[i - 1]: i -= 1
    start = i
    span_db = (p[imax] - p[start]) if imax > start else 0.0
    if (imax - start) < int(5 * fs) or span_db < 10.0:
        below = np.where(p[:imax + 1] < 8.0)[0]
        start = int(below[-1]) if below.size else 0
    return start


def section_to_cnv(src_cnv, out_cnv):
    """Byte-for-byte sectioned .cnv (step01's section_cnv): keep [onset:end], CRLF preserved, only
    '# nvalues' updated. Returns the output path."""
    src = Path(src_cnv); out = Path(out_cnv)
    lines = src.read_bytes().splitlines(keepends=True)
    end_idx = None
    for k, line in enumerate(lines):
        if line.strip().upper().startswith(b"*END*"): end_idx = k; break
    if end_idx is None: raise ValueError(f"No *END* in {src.name}")
    header = lines[:end_idx + 1]
    data_rows = [ln for ln in lines[end_idx + 1:] if ln.strip() != b""]
    if not data_rows: raise ValueError(f"{src.name} has no data rows")
    header_text = [ln.decode("latin-1", "replace").rstrip("\r\n") for ln in header]
    pcol = _cnv_pressure_column(header_text)
    if pcol is None: raise ValueError(f"no pressure column in {src.name}")
    fs = _cnv_sample_rate(header_text)
    pressure = np.full(len(data_rows), np.nan)
    for k, ln in enumerate(data_rows):
        parts = ln.decode("latin-1", "replace").split()
        if pcol < len(parts):
            try: pressure[k] = float(parts[pcol])
            except ValueError: pressure[k] = np.nan
    start = _descent_onset(pressure, fs)
    # Shallow-cast floor, mirrored from step01: never trim below SECTION_SHALLOW_FLOOR_DB on a
    # shallow cast; the motion onset still wins when it trims less (min of the two indices).
    if np.isfinite(pressure).any() and float(np.nanmax(pressure)) < SHALLOW_CAST_MAX_DB:
        _imax = int(np.nanargmax(pressure))
        _at_floor = np.where(pressure[:_imax + 1] >= SECTION_SHALLOW_FLOOR_DB)[0]
        _cap = int(_at_floor[0]) if _at_floor.size else 0
        start = min(start, _cap)
    kept = data_rows[start:]
    nval_re = re.compile(rb"(#\s*nvalues\s*=\s*)(\d+)", re.I)
    new_header = [nval_re.sub(lambda mm: mm.group(1) + str(len(kept)).encode("ascii"), ln)
                  if nval_re.search(ln) else ln for ln in header]
    with open(out, "wb") as fh:
        fh.writelines(new_header); fh.writelines(kept)
    return out


P_SUBMERGE_DB = 0.5   # pressure (db) at/above which the package counts as in the water

def _pressure_fs(cnv_path):
    """(pressure array, fs) from a .cnv, using the same header helpers as Section."""
    lines = Path(cnv_path).read_bytes().splitlines(keepends=True)
    end_idx = next((k for k, ln in enumerate(lines) if ln.strip().upper().startswith(b"*END*")), None)
    if end_idx is None: return None, SECTION_DEFAULT_FS
    header = [l.decode("latin-1", "replace").rstrip("\r\n") for l in lines[:end_idx + 1]]
    pcol = _cnv_pressure_column(header); fs = _cnv_sample_rate(header)
    if pcol is None: return None, fs
    rows = [l for l in lines[end_idx + 1:] if l.strip() != b""]
    p = np.full(len(rows), np.nan)
    for k, ln in enumerate(rows):
        parts = ln.decode("latin-1", "replace").split()
        if pcol < len(parts):
            try: p[k] = float(parts[pcol])
            except ValueError: pass
    return p, fs


def soak_seconds(cnv_path):
    """Surface soak duration (s) from a PRE-section cnv (the Cell TM output still has the soak): time
    between first submersion (P >= P_SUBMERGE_DB) and the descent onset Section uses. The SAME
    _descent_onset() as Section defines the boundary, so this is exactly the in-water time Section
    trims off the top of the cast. Returns (soak_s, n_soak_scans)."""
    try:
        p, fs = _pressure_fs(cnv_path)
    except Exception:
        return np.nan, 0
    if p is None or not np.isfinite(p).any() or not fs or fs <= 0:
        return np.nan, 0
    onset = _descent_onset(p, fs)
    sub = np.where(p >= P_SUBMERGE_DB)[0]
    first_in = int(sub[0]) if sub.size else 0
    n_soak = max(onset - first_in, 0)
    return (n_soak / fs, int(n_soak))


def run_chain(cast, verbose=True):
    """Run DatCnv(baseline)->BinAvg, capturing each stage. Returns OrderedDict stage->{cnv, df}."""
    L1 = DATCNV_DIR / f"{cast}.cnv"
    if not L1.exists():
        if verbose: print(f"  {cast}: no DatCnv cnv at {L1} - run step01 first.")
        return None
    S = OrderedDict()
    S["datcnv"] = dict(cnv=L1, df=parse_cnv_raw(L1))
    f  = run_sbe("FilterW.exe",   PSA["filter"], L1, f"{cast}_d1filt");  S["filter"]  = dict(cnv=f,  df=parse_cnv_raw(f))
    a  = run_sbe("AlignCTDW.exe", PSA["align"],  f,  f"{cast}_d2align"); S["align"]   = dict(cnv=a,  df=parse_cnv_raw(a))
    ct = run_sbe("CellTMW.exe",   PSA["celltm"], a,  f"{cast}_d3ctm");   S["celltm"]  = dict(cnv=ct, df=parse_cnv_raw(ct))
    sec = section_to_cnv(ct, WORK / f"{cast}_d4sect.cnv") if ct else None
    S["section"] = dict(cnv=sec, df=parse_cnv_raw(sec))
    # Shallow-cast Loop Edit, mirrored from step01: a cast shallower than SHALLOW_CAST_MAX_DB uses the
    # fixed-min-velocity PSA (06b). p_max is read from the pre-section Cell TM cnv.
    _lep = PSA["loopedit"]
    if ct:
        _pp, _ = _pressure_fs(ct)
        if _pp is not None and np.isfinite(_pp).any() and float(np.nanmax(_pp)) < SHALLOW_CAST_MAX_DB:
            _lep = PSA.get("loopedit_shallow", PSA["loopedit"])
    le = run_sbe("LoopEditW.exe", _lep, sec, f"{cast}_d5loop") if sec else None
    S["loopedit"] = dict(cnv=le, df=parse_cnv_raw(le))
    de = run_sbe("DeriveW.exe",   PSA["derive"], le, f"{cast}_d6derive") if le else None
    S["derive"] = dict(cnv=de, df=parse_cnv_raw(de))
    ba = run_sbe("BinAvgW.exe",   PSA["binavg"], de, f"{cast}_d7binavg") if de else None
    S["binavg"] = dict(cnv=ba, df=parse_cnv_raw(ba))
    if verbose:
        print("  stages:", ", ".join(f"{k}={0 if S[k]['df'] is None else len(S[k]['df'])}" for k in S))
    return S


_CHAIN_CACHE = {}
def ensure_chain(cast):
    """Run (or reuse) the full chain for a cast and cache it, so the interactive explorer can cover
    every cast without re-processing one you have already opened."""
    if cast not in _CHAIN_CACHE:
        print(f"processing {cast} (first view) ...")
        _CHAIN_CACHE[cast] = run_chain(cast, verbose=False)
    return _CHAIN_CACHE[cast]


## 4. Per-stage metrics (tidy ledger)

One tidy table per cast, long form `(cast, stage, track, metric, variable, value, unit)`, so both the
CSV and the figures read from the same source.

In [ ]:
def _row(cast, stage, track, metric, value, variable="", unit=""):
    return dict(cast=cast, stage=stage, track=track, metric=metric,
                variable=variable, value=(None if value is None else value), unit=unit)


def compute_metrics(cast, S):
    """Return a list of tidy metric rows for one cast's captured stages S."""
    rows = []
    def n_of(k):
        d = S.get(k, {}).get("df"); return 0 if d is None else len(d)

    # ---------- ROW TRACK ----------
    base = S["datcnv"]["df"]
    n0 = n_of("datcnv")
    rows.append(_row(cast, "datcnv", "row", "scans", n0, unit="scans"))
    if base is not None:
        p = num(base, col_P(base))
        if p is not None and np.isfinite(p).any():
            rows.append(_row(cast, "datcnv", "row", "p_min", float(np.nanmin(p)), unit="dbar"))
            rows.append(_row(cast, "datcnv", "row", "p_max", float(np.nanmax(p)), unit="dbar"))
        # bad-flag count per channel (scans set to NaN by the sentinel mask)
        badP = int(np.sum(~np.isfinite(p))) if p is not None else 0
        rows.append(_row(cast, "datcnv", "row", "badflag_P", badP, variable="P", unit="scans"))

    # value-modifying stages keep the scan count; record it and any edge loss
    prev = "datcnv"
    for k in ("filter", "align", "celltm"):
        nk = n_of(k)
        rows.append(_row(cast, k, "row", "scans", nk, unit="scans"))
        lost = n_of(prev) - nk
        rows.append(_row(cast, k, "row", "edge_scans_lost", max(lost, 0), unit="scans"))
        prev = k

    # section removes soak+dip
    n_ct, n_sec = n_of("celltm"), n_of("section")
    rows.append(_row(cast, "section", "row", "scans", n_sec, unit="scans"))
    rows.append(_row(cast, "section", "row", "removed", max(n_ct - n_sec, 0), unit="scans"))
    rows.append(_row(cast, "section", "row", "removed_pct",
                     (100*(n_ct-n_sec)/n_ct) if n_ct else np.nan, unit="%"))
    # surface soak duration (s): measured on the PRE-section Cell TM cnv, from first submersion to the
    # SAME descent onset Section uses, so it is exactly the in-water time Section trims off the top.
    _ct_cnv = S.get("celltm", {}).get("cnv")
    _soak_s, _ = soak_seconds(_ct_cnv) if _ct_cnv else (np.nan, 0)
    rows.append(_row(cast, "section", "row", "soak_seconds", _soak_s, unit="s"))
    _pcol_b = col_P(base) if base is not None else None
    _pmax_cast = float(np.nanmax(num(base, _pcol_b))) if _pcol_b else np.nan
    _is_shallow = bool(np.isfinite(_pmax_cast) and _pmax_cast < SHALLOW_CAST_MAX_DB)
    rows.append(_row(cast, "section", "row", "shallow_cast", 1.0 if _is_shallow else 0.0, unit="bool"))
    rows.append(_row(cast, "loopedit", "row", "fixed_min_velocity", 1.0 if _is_shallow else 0.0, unit="bool"))

    # loop edit flags (marks, does not delete)
    led = S["loopedit"]["df"]
    rows.append(_row(cast, "loopedit", "row", "scans", n_of("loopedit"), unit="scans"))
    if led is not None and "flag" in led.columns:
        fl = pd.to_numeric(led["flag"], errors="coerce").fillna(0) != 0
        rows.append(_row(cast, "loopedit", "row", "flagged", int(fl.sum()), unit="scans"))
        rows.append(_row(cast, "loopedit", "row", "flagged_pct",
                         float(100*fl.mean()) if len(fl) else np.nan, unit="%"))

    # derive adds variables
    rows.append(_row(cast, "derive", "row", "scans", n_of("derive"), unit="scans"))

    # bin average: scans in -> bins out, scans-per-bin, within-bin std (computed from derive stage)
    de = S["derive"]["df"]; nbins = n_of("binavg")
    rows.append(_row(cast, "binavg", "row", "bins", nbins, unit="bins"))
    if de is not None:
        p = num(de, col_P(de))
        if p is not None and np.isfinite(p).any():
            depth = p  # dbar ~ m on the shelf; bin by ~1 m
            b = np.round(depth / BIN_DB) * BIN_DB
            dd = pd.DataFrame({"b": b, "T": num(de, col_T(de))})
            cS, _ = cond_Sm(de)
            dd["S"] = salinity_of(de)
            grp = dd.groupby("b")
            spb = grp.size().to_numpy()
            rows.append(_row(cast, "binavg", "row", "scans_in", int(len(de)), unit="scans"))
            rows.append(_row(cast, "binavg", "row", "scans_per_bin_mean", float(np.mean(spb)), unit="scans"))
            rows.append(_row(cast, "binavg", "row", "scans_per_bin_min", float(np.min(spb)), unit="scans"))
            rows.append(_row(cast, "binavg", "row", "scans_per_bin_max", float(np.max(spb)), unit="scans"))
            for v in ("T", "S"):
                wstd = grp[v].std().to_numpy()
                rows.append(_row(cast, "binavg", "value", "within_bin_std",
                                 float(np.nanmean(wstd)), variable=v,
                                 unit=("degC" if v == "T" else "PSU")))

    # ---------- VALUE TRACK ----------
    def pack(before, after, var, col_before=None, col_after=None, to_cond=False):
        if before is None or after is None: return
        if to_cond:
            xb, _ = cond_Sm(before); xa, _ = cond_Sm(after)
        else:
            xb = num(before, col_before); xa = num(after, col_after)
        if xb is None or xa is None: return
        pr = num(before, col_P(before))
        d = value_delta(xb, xa, pressure=pr, res=RES.get(var))
        rows.append(_row(cast, stg, "value", "rms_delta", d["rms"], variable=var,
                         unit={"P": "dbar", "T": "degC", "C": "S/m", "S": "PSU"}[var]))
        rows.append(_row(cast, stg, "value", "nrms_delta", d["nrms"], variable=var, unit="xSD"))
        rows.append(_row(cast, stg, "value", "pct_moved", d["pct_moved"], variable=var, unit="%"))
        rows.append(_row(cast, stg, "value", "max_abs_delta", d["max_abs"], variable=var,
                         unit={"P": "dbar", "T": "degC", "C": "S/m", "S": "PSU"}[var]))
        rows.append(_row(cast, stg, "value", "p_at_max", d["p_at_max"], variable=var, unit="dbar"))

    # Filter vs DatCnv: P, T, C deltas + HF variance reduction
    stg = "filter"; b, a = S["datcnv"]["df"], S["filter"]["df"]
    if b is not None and a is not None:
        pack(b, a, "P", col_P(b), col_P(a)); pack(b, a, "T", col_T(b), col_T(a)); pack(b, a, "C", to_cond=True)
        for var, getter in (("P", lambda d: num(d, col_P(d))),
                            ("T", lambda d: num(d, col_T(d))),
                            ("C", lambda d: cond_Sm(d)[0])):
            sb, sa = dxdt_std(getter(b)), dxdt_std(getter(a))
            red = 100*(1 - sa/sb) if (sb and np.isfinite(sb) and sb > 0) else np.nan
            rows.append(_row(cast, "filter", "value", "hf_var_reduction", red, variable=var, unit="%"))

    # Align vs Filter: raw deltas (reported) + salinity-spike reduction (headline)
    stg = "align"; b, a = S["filter"]["df"], S["align"]["df"]
    if b is not None and a is not None:
        pack(b, a, "T", col_T(b), col_T(a)); pack(b, a, "C", to_cond=True)
        Sb, Sa = salinity_of(b), salinity_of(a)
        if Sb is not None and Sa is not None:
            nmin = min(len(Sb), len(Sa))
            sb, sa = spike_index(Sb[:nmin]), spike_index(Sa[:nmin])
            red = 100*(1 - sa/sb) if (sb and np.isfinite(sb) and sb > 0) else np.nan
            rows.append(_row(cast, "align", "value", "salinity_spike_reduction", red, variable="S", unit="%"))

    # Cell TM vs Align: C and S delta, whole-profile and thermocline-restricted
    stg = "celltm"; b, a = S["align"]["df"], S["celltm"]["df"]
    if b is not None and a is not None:
        pack(b, a, "C", to_cond=True)
        Sb, Sa = salinity_of(b), salinity_of(a)
        if Sb is not None and Sa is not None:
            pr = num(b, col_P(b))
            d_all = value_delta(Sb, Sa, pressure=pr, res=RES.get("S", 0.001))
            rows.append(_row(cast, "celltm", "value", "rms_delta", d_all["rms"], variable="S", unit="PSU"))
            rows.append(_row(cast, "celltm", "value", "nrms_delta", d_all["nrms"], variable="S", unit="xSD"))
            # thermocline-restricted
            T = num(b, col_T(b)); P = num(b, col_P(b))
            nmin = min(len(Sb), len(Sa), len(T), len(P))
            tm = thermocline_mask(T[:nmin], P[:nmin])
            dS = (np.asarray(Sa[:nmin]) - np.asarray(Sb[:nmin]))
            mm = tm & np.isfinite(dS)
            rows.append(_row(cast, "celltm", "value", "rms_delta_thermocline",
                             float(np.sqrt(np.nanmean(dS[mm]**2))) if mm.sum() > 2 else np.nan,
                             variable="S", unit="PSU"))

    # Derive: density inversions on the downcast
    de = S["derive"]["df"]
    if de is not None:
        cS, _ = cond_Sm(de); T = num(de, col_T(de)); P = num(de, col_P(de))
        if cS is not None and T is not None and P is not None:
            ninv, maxinv = density_inversions(cS, T, P)
            rows.append(_row(cast, "derive", "value", "density_inversions", ninv, variable="sigma_theta", unit="count"))
            rows.append(_row(cast, "derive", "value", "max_inversion", maxinv, variable="sigma_theta", unit="kg/m3"))
    return rows

## 5. Figures (per-cast waterfall + value-change, and the fleet summary)

In [ ]:
def _val(led, stage, metric, variable=None):
    q = led[(led["stage"] == stage) & (led["metric"] == metric)]
    if variable is not None: q = q[q["variable"] == variable]
    if q.empty: return np.nan
    return q["value"].iloc[0]


def waterfall_figure(cast, led, out=None):
    """Scan population across the chain: bars of scans per stage; Loop Edit split kept/flagged;
    Section drop annotated; Bin Average shown as bins."""
    stages = ["datcnv", "filter", "align", "celltm", "section", "loopedit", "derive"]
    scans = [_val(led, s, "scans") for s in stages]
    fig, ax = new_static_fig(figsize=(9.5, 4.8))
    x = np.arange(len(stages) + 1)
    # linear stages
    for i, s in enumerate(stages):
        n = scans[i]
        if not np.isfinite(n): n = 0
        if s == "loopedit":
            fl = _val(led, "loopedit", "flagged"); fl = 0 if not np.isfinite(fl) else fl
            ax.bar(i, n - fl, color=C_KEEP)
            ax.bar(i, fl, bottom=n - fl, color=C_FLAG, label="flagged")
            ax.annotate(f"{fl:.0f} flagged ({_val(led,'loopedit','flagged_pct'):.0f}%)",
                        (i, n), textcoords="offset points", xytext=(0, 3),
                        ha="center", va="bottom", fontsize=7, color="#7a5c00")
        else:
            ax.bar(i, n, color=C_KEEP, label="kept" if s == "datcnv" else None)
            ax.annotate(f"{n:.0f}", (i, n), textcoords="offset points", xytext=(0, 3),
                        ha="center", va="bottom", fontsize=7.5)
    # section removal annotation (between celltm and section)
    rem = _val(led, "section", "removed"); rempct = _val(led, "section", "removed_pct")
    if np.isfinite(rem):
        ax.annotate(f"-{rem:.0f} soak+dip ({rempct:.0f}%)", (4, scans[4]),
                    textcoords="offset points", xytext=(0, 18),
                    ha="center", va="bottom", fontsize=7, color=C_REMOVE)
    # bin average as bins (separate scale note)
    bins = _val(led, "binavg", "bins"); bins = 0 if not np.isfinite(bins) else bins
    ax.bar(len(stages), bins, color="#777777")
    ax.annotate(f"{bins:.0f} bins", (len(stages), bins), ha="center", va="bottom", fontsize=7.5)
    ax.set_xticks(x); ax.set_xticklabels([STAGE_LABEL[s] for s in stages] + ["Bin Avg\n(1 m)"], fontsize=8)
    ax.set_ylabel("scans"); ax.set_title(f"{cast}  -  scan population through the processing chain")
    ax.legend(loc="upper right", fontsize=7.5); _despine(ax)
    ax.set_ylim(top=ax.get_ylim()[1]*1.10)
    fig.tight_layout()
    return finish(fig, out or (QC_PLOT_DIR / f"{cast}__audit_waterfall.png"))


def change_by_stage_figure(cast, led, out=None):
    """Normalised value change (RMS delta / profile SD) by variable at Filter, Align, Cell TM,
    plus the targeted diagnostics (HF reduction, spike reduction, inversions) as a side panel."""
    stages = ["filter", "align", "celltm"]; vars_ = ["P", "T", "C", "S"]
    fig, (ax, ax2) = new_static_fig(1, 2, figsize=(10.5, 4.6),
                                    gridspec_kw={"width_ratios": [2.0, 1.0]}, squeeze=True)
    w = 0.2
    for j, v in enumerate(vars_):
        vals = [_val(led, s, "nrms_delta", v) for s in stages]
        vals = [0 if not np.isfinite(x) else x for x in vals]
        ax.bar(np.arange(len(stages)) + (j - 1.5)*w, vals, w, color=VAR_COLOR[v], label=v)
    ax.set_xticks(np.arange(len(stages))); ax.set_xticklabels([STAGE_LABEL[s] for s in stages])
    ax.set_ylabel("normalised change (RMS delta / profile SD)")
    ax.set_title(f"{cast}  -  value change by stage"); ax.legend(title="variable", fontsize=7.5)
    _despine(ax)
    # side panel: headline diagnostics
    labels, vals, cols = [], [], []
    for v in ("P", "T", "C"):
        r = _val(led, "filter", "hf_var_reduction", v)
        if np.isfinite(r): labels.append(f"Filter HF\n{v}"); vals.append(r); cols.append(VAR_COLOR[v])
    sr = _val(led, "align", "salinity_spike_reduction", "S")
    if np.isfinite(sr): labels.append("Align spike\nreduction"); vals.append(sr); cols.append(VAR_COLOR["S"])
    ax2.barh(np.arange(len(labels)), vals, color=cols)
    ax2.set_yticks(np.arange(len(labels))); ax2.set_yticklabels(labels, fontsize=7)
    ax2.set_xlabel("reduction (%)"); ax2.set_title("targeted diagnostics", fontsize=9)
    ax2.axvline(0, color="k", lw=0.5, alpha=0.4); _despine(ax2)
    ninv = _val(led, "derive", "density_inversions", "sigma_theta")
    if np.isfinite(ninv):
        ax2.annotate(f"Derive: {ninv:.0f} density inversions", (0.5, -0.18), xycoords="axes fraction",
                     ha="center", fontsize=7.5, color="#555555")
    fig.tight_layout()
    return finish(fig, out or (QC_PLOT_DIR / f"{cast}__audit_change_by_stage.png"))


def summary_figure(agg, out=None):
    """Fleet summary: median + range (min..max) across casts for the headline metrics."""
    fig, (axr, axv) = new_static_fig(1, 2, figsize=(11.0, 4.6), squeeze=True)
    # row attrition: section removed_pct, loopedit flagged_pct
    rlabels = ["Section\nremoved %", "Loop Edit\nflagged %"]
    rkeys = [("section", "removed_pct", ""), ("loopedit", "flagged_pct", "")]
    for i, (st, mt, vv) in enumerate(rkeys):
        med, lo, hi = agg_stat(agg, st, mt, vv)
        axr.errorbar(med, i, xerr=[[med-lo], [hi-med]], fmt="o", color=C_REMOVE, capsize=3)
        axr.annotate(f"{med:.0f}% [{lo:.0f}, {hi:.0f}]", (med, i), textcoords="offset points",
                     xytext=(6, 6), fontsize=7)
    axr.set_yticks(range(len(rlabels))); axr.set_yticklabels(rlabels)
    axr.set_xlabel("percent of scans"); axr.set_title("row attrition (median, range)"); _despine(axr)
    axr.set_ylim(-0.5, len(rlabels)-0.5)
    # value change: nrms_delta per variable at each value stage
    combos = [("filter", "C"), ("filter", "T"), ("align", "C"), ("celltm", "S")]
    ylab = [f"{STAGE_LABEL[s]}\n{v}" for s, v in combos]
    for i, (st, v) in enumerate(combos):
        med, lo, hi = agg_stat(agg, st, "nrms_delta", v)
        axv.errorbar(med, i, xerr=[[max(med-lo,0)], [max(hi-med,0)]], fmt="s", color=VAR_COLOR[v], capsize=3)
        axv.annotate(f"{med:.2f} [{lo:.2f}, {hi:.2f}]", (med, i), textcoords="offset points",
                     xytext=(6, 6), fontsize=7)
    axv.set_yticks(range(len(combos))); axv.set_yticklabels(ylab)
    axv.set_xlabel("normalised change (RMS delta / profile SD)")
    axv.set_title("value change (median, range)"); _despine(axv)
    axv.set_ylim(-0.5, len(combos)-0.5)
    fig.suptitle(f"{CRUISE_ID}  -  stage-change summary across {int(agg['cast'].nunique())} casts",
                 fontsize=11, y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    return finish(fig, out or (QC_PLOT_DIR / f"{CRUISE_ID}__audit_summary.png"))


def agg_stat(agg, stage, metric, variable=""):
    q = agg[(agg["stage"] == stage) & (agg["metric"] == metric)]
    if variable != "": q = q[q["variable"] == variable]
    v = pd.to_numeric(q["value"], errors="coerce").dropna()
    if v.empty: return np.nan, np.nan, np.nan
    return float(v.median()), float(v.min()), float(v.max())


# ---------------------------------------------------------------------------
# Interactive stage explorer support: one variable's profile at EACH stage, downcast vs upcast.
# ---------------------------------------------------------------------------
PROFILE_VARS = ["Salinity", "Temperature", "Conductivity", "Sigma-theta", "Oxygen"]
VAR_UNIT = {"Salinity": "PSU", "Temperature": "deg C", "Conductivity": "S/m",
            "Sigma-theta": "kg/m^3", "Oxygen": "umol/L or V"}


def _stage_var_xy(df, var):
    """Return (x-values, pressure) for `var` from a stage DataFrame, computing S / sigma-theta from
    C,T,P so the same variable is available at EVERY stage (not only after Derive). (None, None) if
    the variable cannot be formed at this stage."""
    if df is None or len(df) == 0:
        return None, None
    p = num(df, col_P(df))
    if p is None:
        return None, None
    if var == "Temperature":
        x = num(df, col_T(df))
    elif var == "Conductivity":
        x, _ = cond_Sm(df)
    elif var == "Salinity":
        x = salinity_of(df)
    elif var == "Sigma-theta":
        cS, _ = cond_Sm(df); t = num(df, col_T(df))
        x = sigma_theta(cS, t, p) if (cS is not None and t is not None) else None
    elif var == "Oxygen":
        oc = pick(df, "sbeox", "oxygen", "oxsol", "oxsat")
        x = num(df, oc) if oc else None
    else:
        x = None
    return x, p


def _render_stage_profiles(fig, axes, cast, S, var):
    """Draw a 2x4 grid: one panel per stage, `var` (x) vs pressure (y, surface at top), with the
    downcast and upcast drawn separately and loop-edit flagged scans marked. Shared by the live
    explorer and the static per-cast export."""
    from matplotlib.lines import Line2D
    axes = np.asarray(axes).ravel()
    for k, stage in enumerate(STAGE_ORDER):
        ax = axes[k]; d = S.get(stage, {}).get("df")
        ax.set_title(STAGE_LABEL[stage], fontsize=9.5)
        if d is None or len(d) == 0:
            ax.text(0.5, 0.5, "(no data)", ha="center", va="center", transform=ax.transAxes,
                    color="#aaaaaa", fontsize=8); _despine(ax); continue
        x, pr = _stage_var_xy(d, var)
        if x is None:
            ax.text(0.5, 0.5, f"{var}\nnot available", ha="center", va="center",
                    transform=ax.transAxes, color="#aaaaaa", fontsize=8); _despine(ax); continue
        n = min(len(x), len(pr)); x = np.asarray(x[:n], float); pr = np.asarray(pr[:n], float)
        imax = int(np.nanargmax(pr)) if np.isfinite(pr).any() else n - 1
        ax.plot(x[:imax + 1], pr[:imax + 1], lw=0.6, color=C_DOWN, label="downcast")
        nd, nu = imax + 1, n - imax - 1
        if stage != "binavg" and nu > 1:                       # 1 m product is downcast only
            ax.plot(x[imax:], pr[imax:], lw=0.7, color=C_UP, alpha=0.9, linestyle="--", label="upcast")
        fl = flag_mask(d)
        if fl is not None and fl.any():
            m = fl.to_numpy()[:n] & np.isfinite(x) & np.isfinite(pr)
            if m.any():
                ax.scatter(x[m], pr[m], s=5, color=C_REMOVE, zorder=5, edgecolors="none")
        ax.annotate(f"n={n}\ndn {nd} / up {nu}", (0.97, 0.03), xycoords="axes fraction",
                    ha="right", va="bottom", fontsize=6.3, color="#777777")
        if k >= 4:
            ax.set_xlabel(f"{var} ({VAR_UNIT.get(var,'')})", fontsize=8)
        if k % 4 == 0:
            ax.set_ylabel("pressure (db)", fontsize=8)
        _despine(ax)
    axes[0].invert_yaxis()                                      # sharey -> invert once (surface at top)
    leg = [Line2D([0], [0], color=C_DOWN, lw=1.4, label="downcast"),
           Line2D([0], [0], color=C_UP, lw=1.4, linestyle="--", label="upcast"),
           Line2D([0], [0], marker="o", color="none", markerfacecolor=C_REMOVE, markersize=5,
                  label="loop-edit flagged")]
    fig.legend(handles=leg, loc="upper center", bbox_to_anchor=(0.5, 0.955),
               fontsize=8, ncol=3, frameon=False)
    fig.suptitle(f"{cast}  -  {var} at each stage  (downcast vs upcast)", fontsize=12, y=0.992)
    fig.text(0.01, 0.005, "Each panel is one processing stage, in order. y-axis is pressure with the "
             "surface at the top. Dark solid = downcast, grey dashed = upcast, red = loop-edit flagged.",
             fontsize=7, color="#666666", va="bottom")
    fig.tight_layout(rect=[0, 0.035, 1, 0.91])
    return True


def stage_profiles_static(cast, S, var, out=None):
    """Static PNG version (detached Agg) for a given cast + variable, saved to qc_plots."""
    fig, axes = new_static_fig(2, 4, figsize=(12.0, 7.0), sharey=True)
    _render_stage_profiles(fig, axes, cast, S, var)
    safe = var.lower().replace("-", "").replace(" ", "")
    return finish(fig, out or (QC_PLOT_DIR / f"{cast}__stage_profiles_{safe}.png"))



## 6. Detail casts: full walk-through (waterfall + value change + ledger CSV)

In [ ]:
detail_ledgers = {}
check_psas()                                    # warn up front if any .psa is missing
for cast in DETAIL_CASTS:
    print(f"\n=== {cast} ===")
    S = ensure_chain(cast)
    if not S:
        continue
    rows = compute_metrics(cast, S)
    led = pd.DataFrame(rows)
    detail_ledgers[cast] = led
    led.to_csv(QC_PLOT_DIR / f"{cast}__stage_change_ledger.csv", index=False, encoding="utf-8-sig")
    waterfall_figure(cast, led)
    change_by_stage_figure(cast, led)
    # compact printed summary
    print("  section removed: {:.0f} ({:.0f}%) | loopedit flagged: {:.0f} ({:.0f}%)".format(
        _val(led, "section", "removed"), _val(led, "section", "removed_pct"),
        _val(led, "loopedit", "flagged"), _val(led, "loopedit", "flagged_pct")))
    print("  filter HF reduction C/P: {:.0f}% / {:.0f}% | align spike reduction: {:.0f}%".format(
        _val(led, "filter", "hf_var_reduction", "C"), _val(led, "filter", "hf_var_reduction", "P"),
        _val(led, "align", "salinity_spike_reduction", "S")))
    print("  ledger ->", (QC_PLOT_DIR / f"{cast}__stage_change_ledger.csv").name)

## 7. Fleet summary across all casts (median + range)

In [ ]:
all_rows = []
for cast in ALL_CASTS:
    S = run_chain(cast, verbose=False)
    if not S:
        print(f"  {cast}: skipped (no DatCnv cnv)"); continue
    try:
        all_rows.extend(compute_metrics(cast, S))
    except Exception as e:
        print(f"  {cast}: metric error: {e}")

if all_rows:
    agg = pd.DataFrame(all_rows)
    agg.to_csv(QC_PLOT_DIR / f"{CRUISE_ID}__stage_change_allcasts.csv", index=False, encoding="utf-8-sig")
    # compact median+range table for the meeting
    def line(stage, metric, variable=""):
        m, lo, hi = agg_stat(agg, stage, metric, variable)
        tag = f"{STAGE_LABEL.get(stage,stage)} {metric}" + (f" [{variable}]" if variable else "")
        print(f"  {tag:<34} median {m:.3g}  range [{lo:.3g}, {hi:.3g}]")
    print(f"\nAcross {agg['cast'].nunique()} casts:")
    line("section", "removed_pct"); line("loopedit", "flagged_pct")
    line("filter", "hf_var_reduction", "C"); line("filter", "hf_var_reduction", "P")
    line("align", "salinity_spike_reduction", "S")
    line("celltm", "nrms_delta", "S"); line("celltm", "rms_delta_thermocline", "S")
    line("binavg", "scans_per_bin_mean")
    summary_figure(agg)
    print("\n  summary CSV ->", (QC_PLOT_DIR / f"{CRUISE_ID}__stage_change_allcasts.csv").name)
else:
    print("No casts processed; check DATCNV_DIR and that step01 has produced L1 cnvs.")

## 8. Interactive stage explorer: upcast vs downcast at every stage (all casts)

Pick a **cast** and a **variable**; the grid redraws with that variable's profile at each of the eight stages, downcast and upcast drawn separately so you can see how each step changes the shape and how the two directions differ. Loop-edit flagged scans are marked red.

**How to use it**

- *cast* lists every cast found in `L1/CTD`. The first time you open a cast it runs the chain for that cast (a few seconds, you will see `processing ...`); after that it is instant.
- *variable*: Salinity and Sigma-theta are computed from C, T, P at every stage, so they are available even before Derive (that is where Filter, Align and Cell TM effects show up most clearly, as reduced salinity spiking).
- By default it renders a clear inline image that **updates every time you change the cast or variable**, and saves each view to `qc_plots/`. This always shows. To try experimental mouse-drag zoom, set `INTERACTIVE_ZOOM = True` in section 1 and Restart Kernel + Run All (needs `ipympl`; if the canvas comes up blank, set it back to False).

What to look for, stage by stage: **DatCnv to Cell TM** still contain the surface soak and equilibration dip at the top of the downcast; **Section** removes them; **Loop Edit** marks the heave/upcast scans in red; **Bin Average** is the 1 m downcast product only (no upcast).

In [ ]:
# ---- interactive stage explorer: one variable's profile at each stage, downcast vs upcast ----
# Pick a cast and a variable; the 2x4 grid shows that variable at every stage (down solid, up dashed,
# loop-edit flags in red). A cast's FIRST view runs the full chain (the "rendering ..." line, a few
# seconds on Windows); it is cached after that, so later views are instant.
# INTERACTIVE_ZOOM = True (section 1) = live zoom canvas (toolbar: box-zoom, pan, home, save) and the
# canvas is resizable from its lower-right corner. If it renders blank, set INTERACTIVE_ZOOM = False
# for the large static image.
EXPLORER_FIGSIZE = (14.5, 8.6)   # large panels (both the live canvas and the static image use this)

try:
    import ipywidgets as W
    from IPython.display import clear_output
    _HAVE_W = True
except Exception:
    _HAVE_W = False
    print("ipywidgets not installed; install: pip install ipywidgets ipympl, then Restart Kernel + Run All.")

def _explorer_png_bytes(cast, S, var):
    """Render the 2x4 stage grid to PNG bytes (detached Agg; no file or backend dependency, so it
    always paints). Also drops a copy into qc_plots when that is writable (best effort)."""
    import io
    fig, axes = new_static_fig(2, 4, figsize=EXPLORER_FIGSIZE, sharey=True)
    _render_stage_profiles(fig, axes, cast, S, var)
    buf = io.BytesIO(); fig.savefig(buf, format="png", dpi=130); buf.seek(0)
    try:
        QC_PLOT_DIR.mkdir(parents=True, exist_ok=True)
        safe = var.lower().replace("-", "").replace(" ", "")
        fig.savefig(QC_PLOT_DIR / f"{cast}__stage_profiles_{safe}.png")
    except Exception:
        pass
    try:
        import matplotlib.pyplot as _plt; _plt.close(fig)
    except Exception:
        pass
    return buf.getvalue()

if _HAVE_W:
    _ecasts = ALL_CASTS if ALL_CASTS else DETAIL_CASTS
    ecast = W.Dropdown(options=_ecasts, value=_ecasts[0], description="cast")
    evar  = W.Dropdown(options=PROFILE_VARS, value="Salinity", description="variable")
    eout  = W.Output()

    if INTERACTIVE_ZOOM:
        # Live, zoomable ipympl canvas (box-zoom, pan, home, save; resizable from the corner). Created
        # ONCE; each update clears it and redraws, then forces a paint. If it renders blank on your
        # Jupyter, set INTERACTIVE_ZOOM = False in section 1 for the large static image below.
        _efig = plt.figure(figsize=EXPLORER_FIGSIZE, dpi=100)
        try: _efig.canvas.header_visible = False
        except Exception: pass
        try: _efig.canvas.resizable = True
        except Exception: pass

        def _edraw(*_):
            with eout:
                clear_output(wait=True); print(f"rendering {ecast.value} ...")
            S = ensure_chain(ecast.value)
            with eout: clear_output(wait=True)
            _efig.clf()
            if not S:
                with eout:
                    print(f"{ecast.value}: could not process (missing DatCnv L1 cnv or a .psa - see section 6).")
                _efig.canvas.draw_idle(); return
            axes = _efig.subplots(2, 4, sharey=True)
            _render_stage_profiles(_efig, axes, ecast.value, S, evar.value)
            try:
                _efig.canvas.draw(); _efig.canvas.flush_events()   # force the paint (ioff suppresses auto-draw)
            except Exception:
                try: _efig.canvas.draw_idle()
                except Exception: pass

        for _w in (ecast, evar): _w.observe(_edraw, "value")
        display(W.VBox([W.HBox([ecast, evar]), eout]))
        display(_efig.canvas)
        _edraw()
    else:
        # Reliable large inline image (no mouse zoom). Rendered as PNG bytes, so it never depends on
        # the matplotlib display backend or on qc_plots existing - it always paints once processing ends.
        def _edraw(*_):
            with eout:
                clear_output(wait=True); print(f"rendering {ecast.value} ...")
                S = ensure_chain(ecast.value)
                clear_output(wait=True)
                if not S:
                    print(f"{ecast.value}: could not process (missing DatCnv L1 cnv or a .psa - see section 6)."); return
                display(Image(data=_explorer_png_bytes(ecast.value, S, evar.value)))

        for _w in (ecast, evar): _w.observe(_edraw, "value")
        display(W.VBox([W.HBox([ecast, evar]), eout])); _edraw()


## 9. Reproducibility check: does the audit match step01's actual products?

QC/QA gate. For each cast this re-runs the chain (section 3) and compares what it reproduced against step01's on-disk L2 products, stage by stage, reporting the largest absolute difference per stage. Because every SBE module is deterministic (same PSA + same input) and the Section step is byte-for-byte identical to step01's, a correct setup prints `MATCH` (max difference ~0) at every stage. Anything else points at a drifted PSA, a different input, or a changed Section.

If a stage shows `no step01 product`, that folder has not been produced yet (run step01), or your step01 writes it elsewhere, edit `STEP01_PRODUCT_DIR` below to match. Set `VERIFY_CASTS = ALL_CASTS` to check the whole fleet.

In [ ]:
# ---- QC: step00d reproduction vs step01's on-disk L2 products ----
# step01 output folders (from step01_run_sbe_processing_v2.py). Edit if your layout differs.
STEP01_PRODUCT_DIR = {
    "filter":   CRUISE_DIR / "L2" / "CTD" / "Filter",
    "align":    CRUISE_DIR / "L2" / "CTD" / "Align_CTD",
    "celltm":   CRUISE_DIR / "L2" / "CTD" / "Cell_tm",
    "section":  CRUISE_DIR / "L2" / "CTD" / "Section",
    "loopedit": CRUISE_DIR / "L2" / "CTD" / "Loop_Edit",
    "derive":   CRUISE_DIR / "L2" / "CTD" / "Derived",
    "binavg":   CRUISE_DIR / "L2" / "CTD" / "CTD_1m_down",
}
REPRO_TOL   = 1e-6               # max abs difference still called a match (SBE output is deterministic)
VERIFY_CASTS = DETAIL_CASTS      # set to ALL_CASTS to check every cast


def _find_product(folder, cast):
    if not folder.exists():
        return None
    for pat in (f"{cast}.cnv", f"{cast}_*.cnv", f"{cast}*.cnv"):
        hits = sorted(folder.glob(pat))
        if hits:
            return hits[0]
    return None


# "Core" (physical) channels are what the audit's conclusions rest on; oxygen (provisional, Tau-
# dependent) and fluorometer/PAR/beam are auxiliary. We judge MATCH on the CORE channels and report
# the aux delta separately, so a Tau or fluorometer difference does not masquerade as a pipeline error.
_KEY_TOKENS = ("prdm", "prm", "prsm", "t090c", "tv290c", "c0s/m", "cond0s/m", "sal", "sigma",
               "density", "dens", "depsm", "depfm", "potemp", "ptmpc", "ptempc")
def _is_core(col):
    cl = col.lower()
    return any(tok in cl for tok in _KEY_TOKENS)


def verify_cast(cast, S):
    rows = []
    for stage in ("filter", "align", "celltm", "section", "loopedit", "derive", "binavg"):
        got = S.get(stage, {}).get("df")
        prod = _find_product(STEP01_PRODUCT_DIR[stage], cast)
        if got is None:
            rows.append((stage, "-", "no reproduced df", "", "", "")); continue
        if prod is None:
            rows.append((stage, f"{len(got)} vs -", "no step01 product", "", "", "")); continue
        ref = parse_cnv_raw(prod)
        if ref is None:
            rows.append((stage, f"{len(got)} vs ?", "unreadable product", "", "", "")); continue
        shared = [c for c in got.columns if c in ref.columns]
        n = min(len(got), len(ref))
        core_worst = 0.0; all_worst = 0.0; all_col = ""
        for c in shared:
            a = pd.to_numeric(got[c], errors="coerce").to_numpy()[:n]
            b = pd.to_numeric(ref[c], errors="coerce").to_numpy()[:n]
            m = np.isfinite(a) & np.isfinite(b)
            if not m.any(): continue
            d = float(np.max(np.abs(a[m] - b[m])))
            if d > all_worst: all_worst, all_col = d, c
            if _is_core(c) and d > core_worst: core_worst = d
        same_rows = (len(got) == len(ref))
        if not same_rows:
            verdict = "ROWS DIFFER"
        elif core_worst <= REPRO_TOL and all_worst <= REPRO_TOL:
            verdict = "MATCH"
        elif core_worst <= REPRO_TOL:
            verdict = "CORE MATCH (aux differ)"
        else:
            verdict = "CORE DIFFERS"
        rows.append((stage, f"{len(got)} vs {len(ref)}", verdict,
                     f"{core_worst:.2e}", f"{all_worst:.2e}", all_col))
    return rows


print(f"Reproducibility vs step01 L2 products (tol={REPRO_TOL:g}). MATCH = reproduces step01 exactly.\n")
_hdr = f"{'stage':<10}{'rows repro vs step01':<22}{'verdict':<24}{'core max':<11}{'all max':<11}worst col"
_all_match = True
for cast in VERIFY_CASTS:
    S = ensure_chain(cast)
    print(f"=== {cast} ===")
    if not S:
        print("  not processed (missing L1 cnv or .psa)\n"); _all_match = False; continue
    print(_hdr); print("-" * len(_hdr))
    for r in verify_cast(cast, S):
        if r[2] not in ("MATCH", "CORE MATCH (aux differ)"): _all_match = False
        print(f"{r[0]:<10}{r[1]:<22}{r[2]:<24}{r[3]:<11}{r[4]:<11}{r[5]}")
    print()
print("Core (physical) channels MATCH step01 at every stage." if _all_match else
      "Some stages differ on CORE channels or row count (see above) - investigate before using as QC of record.")
print("Note: 'CORE MATCH (aux differ)' = P/T/C/salinity/density reproduce exactly; only auxiliary"
      " channels (oxygen Tau, fluorometer/PAR/beam) differ, which the audit does not analyse.")
